##
Workflow

Dirty CSV Files
      ↓
Microsoft Fabric Workspace
      ↓
OneLake
      ↓
Lakehouse
      ↓
Fabric Notebook
      ↓
Clean / Curated Data
      ↓
Power BI

##
1.1
Loading the raw datasets

In [ ]:
import pandas as pd

users_raw = pd.read_csv(r"D:\old\MAD1-PROJECT\data\raw\raw_users.csv")
campaigns_raw = pd.read_csv(r"D:\old\MAD1-PROJECT\data\raw\raw_campaigns.csv")
ad_requests_raw = pd.read_csv(r"D:\old\MAD1-PROJECT\data\raw\raw_ad_requests.csv")

In [ ]:
#Shape
print("Users shape:", users_raw.shape)
print("Campaigns shape:", campaigns_raw.shape)
print("Ad Requests shape:", ad_requests_raw.shape)

##
1.2
First data-quality check

In [ ]:
users_raw.info()

In [ ]:
campaigns_raw.info()

In [ ]:
ad_requests_raw.info()

##
1.3
Check missing values

In [ ]:
print("Users missing values:\n", users_raw.isnull().sum())

In [ ]:
print("Campaigns missing values:\n", campaigns_raw.isnull().sum())

In [ ]:
print("Ad Requests missing values:\n", ad_requests_raw.isnull().sum())

In [ ]:
print("Users duplicates:", users_raw.duplicated().sum())

In [ ]:
print("Campaign duplicates:", campaigns_raw.duplicated().sum())

In [ ]:
print("Ad Request duplicates:", ad_requests_raw.duplicated().sum())

##
1.5
Categorical Inconsistencies

In [ ]:
print("User Roles:")
print(users_raw["role"].value_counts(dropna=False))

In [ ]:
print("\nCampaign Visibility:")
print(campaigns_raw["visibility"].value_counts(dropna=False))

In [ ]:

print("\nAd Request Status:")
print(ad_requests_raw["status"].value_counts(dropna=False))

##
1.6
Outliers

In [ ]:
print("Campaign budget statistics:")
print(campaigns_raw["budget"].describe())

In [ ]:
print("\nPayment Amount Statistics:")
print(ad_requests_raw["payment_amount"].describe())

In [ ]:
# Payment amount outliers

Q1_payment = ad_requests_raw["payment_amount"].quantile(0.25)
Q3_payment = ad_requests_raw["payment_amount"].quantile(0.75)

IQR_payment = Q3_payment - Q1_payment

lower_payment = Q1_payment - 1.5 * IQR_payment
upper_payment = Q3_payment + 1.5 * IQR_payment

print("Payment IQR:", IQR_payment)
print("Lower Bound:", lower_payment)
print("Upper Bound:", upper_payment)

print("\nPayment Amount Outliers:")
print(
    ad_requests_raw[
        (ad_requests_raw["payment_amount"] < lower_payment) |
        (ad_requests_raw["payment_amount"] > upper_payment)
    ][["id", "payment_amount"]]
)

In [ ]:
#campaign budget outliers

Q1_budget = campaigns_raw["budget"].quantile(0.25)
Q3_budget = campaigns_raw["budget"].quantile(0.75)

IQR_budget = Q3_budget - Q1_budget

lower_budget = Q1_budget - 1.5 * IQR_budget
upper_budget = Q3_budget + 1.5 * IQR_budget

print("Budget IQR:", IQR_budget)
print("Lower Bound:", lower_budget)
print("Upper Bound:", upper_budget)

print("\nCampaign Budget Outliers:")
print(
    campaigns_raw[
        (campaigns_raw["budget"] < lower_budget) |
        (campaigns_raw["budget"] > upper_budget)
    ][["id", "budget"]]
)

##
1.7
Invalid numeric values

In [ ]:
print("\nNegative Campaign Budgets:")
print(campaigns_raw[campaigns_raw["budget"] < 0][["id", "budget"]])

In [ ]:
print("\nNegative Payment Amounts:")
print(ad_requests_raw[ad_requests_raw["payment_amount"] < 0][["id", "payment_amount"]])

##
1.8
Date Problems

In [ ]:
print("Sample Start Dates:")
print(campaigns_raw["start_date"].head(15))

print("\nSample End Dates:")
print(campaigns_raw["end_date"].head(15))

In [ ]:
start_dates = pd.to_datetime(
    campaigns_raw["start_date"],
    errors="coerce"
)

end_dates = pd.to_datetime(
    campaigns_raw["end_date"],
    errors="coerce"
)

print("Invalid start dates:", start_dates.isna().sum())
print("Invalid end dates:", end_dates.isna().sum())

In [ ]:
invalid_date_order = campaigns_raw[
    end_dates < start_dates
]

print("End date before start date:")
print(
    invalid_date_order[
        ["id", "start_date", "end_date"]
    ]
)
#other might be invalid dates, but we are not checking for that here. 
# We are only checking for the order of the dates.
#after we convert into correct datetime format, we will get full invalid dates 

## Data Quality Summary

The raw synthetic datasets contain several data-quality issues that need
to be addressed before data cleaning, transformation, and dashboard development.

### Users

- 10,050 total raw records
- 100 missing usernames
- 76 missing emails
- 50 exact duplicate records
- Inconsistent capitalization and whitespace in the role column

### Campaigns

- 5,050 total raw records
- 101 missing descriptions
- 61 missing budgets
- 50 exact duplicate records
- Inconsistent capitalization and whitespace in visibility
- 30 negative budget values
- 25 high-value budget outliers identified using the IQR upper-bound method
- Mixed date formats are present and require format-aware parsing
- Initial `pd.to_datetime()` conversion produced 201 unparseable start
  dates and 101 unparseable end dates; these results may be caused by
  mixed date formats and should not be treated as confirmed invalid dates
  until format-aware parsing is performed
- 78 records have an end date earlier than the start date

### Ad Requests

- 100,500 total raw records
- 2,515 missing messages
- 2,504 missing payment amounts
- 1,460 missing statuses
- 500 exact duplicate records
- Inconsistent capitalization, whitespace, and misspellings in status
- 301 negative payment values
- 278 statistical IQR outliers based on the calculated IQR bounds;
  these include some negative payment values, so negative payments should
  be handled separately as business-rule violations
- Invalid influencer and sponsor IDs are present and need to be identified
  and flagged during validation

These findings provide the baseline for the next stage of the project:
data cleaning and transformation using PySpark and Microsoft Fabric.